Fine-tuned Chatbot.ipynb

Author: Dr. Jorge Luis Rosas Trigueros
Last modification: 10 may 2026

https://tinyurl.com/45n9muas

In [ ]:
!pip install transformers datasets

In [ ]:
import copy
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

In [ ]:
model_name = "distilgpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# ✅ Fix padding token
tokenizer.pad_token = tokenizer.eos_token

# ✅ Save base model for BEFORE comparison
base_model = copy.deepcopy(model)

In [ ]:
data = [
    "You are a helpful cooking assistant.\nUser: how do I make pancakes?\nBot: To make pancakes, mix flour, milk, eggs, and sugar. Cook on a hot pan until golden.",

    "You are a helpful cooking assistant.\nUser: how to cook rice?\nBot: Rinse the rice, add water, bring to a boil, then simmer until absorbed.",

    "You are a helpful cooking assistant.\nUser: how do I make scrambled eggs?\nBot: Crack eggs into a bowl, whisk them, and cook gently in a pan.",

    "You are a helpful cooking assistant.\nUser: how to bake a cake?\nBot: Mix ingredients and bake at 180°C for 30–40 minutes.",

    "You are a helpful cooking assistant.\nUser: how to make salad?\nBot: Chop vegetables and mix with dressing like olive oil and lemon."
]

dataset = Dataset.from_dict({"text": data})

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, remove_columns=["text"])

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # ✅ required for GPT models
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=4,
    logging_steps=5,
    save_strategy="no",
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator   # ✅ handles padding + labels dynamically
)

trainer.train()

In [ ]:
def generate_response(prompt, model_to_use):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

    output = model_to_use.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],  # ✅ FIX
        max_length=100,
        min_length=10,  # ✅ avoid blank outputs
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id  # ✅ FIX
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()

Prompt categories

 1. Domain-Specific Prompts (Cooking Tasks)
 2. Paraphrase / Rewording Prompts
 3. General Prompt (Open Interpretation)
 4. Edge Case / Noise Input

In [ ]:
test_prompts = [
    # DOMAIN
    "User: how do I cook rice?\nBot:",
    "User: how do I make pancakes?\nBot:",
    "User: how do I fry eggs?\nBot:",
    "User: how to bake cookies?\nBot:",

    # PARAPHRASE
    "User: what is the process for cooking rice?\nBot:",
    "User: how can I prepare pancakes at home?\nBot:",
    "User: what are the steps to fry an egg?\nBot:",

    # GENERAL
    "User: can you help me cook something?\nBot:",
    "User: what is a simple recipe?\nBot:",
    "User: I am hungry, what should I make?\nBot:",

    # EDGE CASES
    "User: ???\nBot:",
    "User: asdfgh\nBot:"
]

prompt_categories = [
    "domain", "domain", "domain", "domain",
    "paraphrase", "paraphrase", "paraphrase",
    "general", "general", "general",
    "edge", "edge"
]

expected_outputs = [
    # DOMAIN
    "rinse rice boil simmer",
    "mix flour milk eggs cook pancakes",
    "heat pan fry egg until cooked",
    "mix ingredients bake cookies oven",

    # PARAPHRASE
    "rice cooking process boil simmer water",
    "prepare pancakes mix ingredients cook pan",
    "steps fry egg heat pan cook egg",

    # GENERAL
    "simple meal recipe easy cooking steps",
    "easy recipe like salad or eggs",
    "make simple food like eggs or rice",

    # EDGE
    "unknown input",
    "unknown input"
]

In [ ]:
print("\n=== BEFORE vs AFTER ===\n")

for prompt in test_prompts:

    before = generate_response(prompt, base_model)
    after = generate_response(prompt, model)

    print("PROMPT:")
    print(prompt)
    print("\nBEFORE:")
    print(before)
    print("\nAFTER:")
    print(after)
    print("=" * 60)

In [ ]:
def categorize_response(expected, predicted):
    predicted = predicted.strip().lower()

    if predicted == "":
        return "EMPTY"

    words = predicted.split()
    if len(words) > 3 and len(set(words)) <= len(words) / 2:
        return "REPETITIVE"

    if any(word in predicted for word in expected.split()):
        if len(predicted.split()) >= 5:
            return "CORRECT"
        else:
            return "PARTIAL"

    return "WRONG"

In [ ]:
from collections import defaultdict

category_results = defaultdict(list)

for prompt, expected, cat in zip(test_prompts, expected_outputs, prompt_categories):

    response = generate_response(prompt, model)
    label = categorize_response(expected, response)

    category_results[cat].append(label)


In [ ]:
from collections import Counter

print("\n=== CATEGORY-WISE CONFUSION ANALYSIS ===\n")

for cat, labels in category_results.items():
    counts = Counter(labels)

    print(f"Category: {cat}")
    for k, v in counts.items():
        print(f"  {k}: {v}")
    print("-" * 40)

In [ ]:
print("Cooking Assistant Chatbot (type 'quit' to exit)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Bot: Goodbye!")
        break

    prompt = f"User: {user_input}\nBot:"
    response = generate_response(prompt, model)

    print("Bot:", response)